In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


In [4]:
df = pd.read_csv("insurance (1).csv")
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
print("Shape:", df.shape)
df.info()


Shape: (1338, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [6]:
df.describe(include="all")


,age,sex,bmi,children,smoker,region,charges
count,1338.000000,1338,1338.000000,1338.000000,1338,1338,1338.000000
unique,NaN,2,NaN,NaN,2,4,NaN
top,NaN,male,NaN,NaN,no,southeast,NaN
freq,NaN,676,NaN,NaN,1064,364,NaN
mean,39.207025,NaN,30.663397,1.094918,NaN,NaN,13270.422265
std,14.049960,NaN,6.098187,1.205493,NaN,NaN,12110.011237
min,18.000000,NaN,15.960000,0.000000,NaN,NaN,1121.873900
25%,27.000000,NaN,26.296250,0.000000,NaN,NaN,4740.287150
50%,39.000000,NaN,30.400000,1.000000,NaN,NaN,9382.033000
75%,51.000000,NaN,34.693750,2.000000,NaN,NaN,16639.912515


## 3. Preprocessing

Check for missing values and inspect the categorical columns.

In [ ]:
df.isnull().sum()


In [ ]:
print("sex:", df["sex"].unique())
print("smoker:", df["smoker"].unique())
print("region:", df["region"].unique())


No missing values are present, so no imputation is required.

## 4. Handle Categorical Data

- `sex` and `smoker` are binary categories -> label-encoded to 0/1.
- `region` has 4 unordered categories -> one-hot encoded (with `drop_first=True` to avoid the dummy-variable trap).

In [ ]:
df["sex"] = df["sex"].map({"male": 1, "female": 0})
df["smoker"] = df["smoker"].map({"yes": 1, "no": 0})

df = pd.get_dummies(df, columns=["region"], drop_first=True)

region_dummy_cols = [c for c in df.columns if c.startswith("region_")]
df[region_dummy_cols] = df[region_dummy_cols].astype(int)

df.head()


## 5. Select Features and Target

In [ ]:
target_col = "charges"
X = df.drop(columns=[target_col])
y = df[target_col]

feature_names = X.columns.tolist()
feature_names


## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])


## 7. Train the Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

for name, coef in zip(feature_names, model.coef_):
    print(f"{name}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")


## 8. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R^2 Score: {r2:.4f}")


## 9. Predictive Function

A reusable function that takes new person details (in their original, human-readable form) and returns the predicted insurance charges.

In [ ]:
def predict_charges(age, sex, bmi, children, smoker, region):
    """
    Predict medical insurance charges for a person.
    sex: "male" or "female"
    smoker: "yes" or "no"
    region: "southwest", "southeast", "northwest", or "northeast"
    """
    sex_encoded = 1 if str(sex).strip().lower() == "male" else 0
    smoker_encoded = 1 if str(smoker).strip().lower() == "yes" else 0

    row = {col: 0 for col in feature_names}
    row["age"] = age
    row["sex"] = sex_encoded
    row["bmi"] = bmi
    row["children"] = children
    row["smoker"] = smoker_encoded

    region_col = f"region_{str(region).strip().lower()}"
    if region_col in row:
        row[region_col] = 1
    # If region == "northeast" (the dropped baseline category),
    # all region_* dummy columns correctly remain 0.

    input_df = pd.DataFrame([row], columns=feature_names)
    prediction = model.predict(input_df)[0]
    return round(prediction, 2)


## 10. Test the Predictive Function

In [ ]:
sample_prediction_1 = predict_charges(
    age=30,
    sex="female",
    bmi=25.0,
    children=1,
    smoker="no",
    region="southeast",
)
print("Predicted charges (Sample 1):", sample_prediction_1)


In [ ]:
sample_prediction_2 = predict_charges(
    age=45,
    sex="male",
    bmi=32.5,
    children=2,
    smoker="yes",
    region="northwest",
)
print("Predicted charges (Sample 2):", sample_prediction_2)
